# EchoVerify: Environment Setup & Exploratory Data Analysis (`setup.ipynb`)

This notebook verifies:
1. Python, PyTorch, GPU, FFmpeg, and audio library readiness.
2. Clean loading of the **ASVspoof 2019 Logical Access (LA)** protocol files and FLAC audio.
3. Official Train / Dev / Eval class and attack ID counts.
4. Exploratory **leading and trailing silence** measurements (bona fide vs. spoof) and sample waveform / log-Mel spectrogram plots.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

required_pkgs = [
    "torchaudio",
    "transformers",
    "librosa",
    "soundfile",
    "scipy",
    "sklearn",
    "yaml",
    "tqdm",
    "streamlit",
]
pkg_status = {p: importlib.util.find_spec(p) is not None for p in required_pkgs}
print("Package check:", pkg_status)
missing = [p for p, ok in pkg_status.items() if not ok]
if missing:
    raise ImportError(f"Missing required packages: {missing}. Run: pip install -r requirements.txt")

ffmpeg_path = shutil.which("ffmpeg")
print(f"FFmpeg path: {ffmpeg_path}")
if ffmpeg_path:
    ver = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True, check=False)
    print(ver.stdout.splitlines()[0] if ver.stdout else "FFmpeg found")
else:
    print("WARNING: ffmpeg not found on PATH. Load ffmpeg module or install it before MP3 experiments.")

In [ ]:
# Resolve repository root and ASVspoof 2019 LA dataset root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

env_root = os.environ.get("ASVSPOOF2019_LA_ROOT")
candidate_roots = [
    Path(env_root) if env_root else None,
    REPO_ROOT / "data" / "LA",
    REPO_ROOT / "data" / "ASVspoof2019_LA",
]

LA_ROOT = None
for cand in candidate_roots:
    if cand and (cand / "ASVspoof2019_LA_cm_protocols").exists():
        LA_ROOT = cand
        break

if LA_ROOT is None:
    raise FileNotFoundError(
        "ASVspoof 2019 LA root not found. Set ASVSPOOF2019_LA_ROOT to the folder containing "
        "ASVspoof2019_LA_cm_protocols, ASVspoof2019_LA_train, ASVspoof2019_LA_dev, and ASVspoof2019_LA_eval."
    )

PROTOCOL_DIR = LA_ROOT / "ASVspoof2019_LA_cm_protocols"
PROTOCOLS = {
    "train": PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt",
    "dev": PROTOCOL_DIR / "ASVspoof2019.LA.cm.dev.trl.txt",
    "eval": PROTOCOL_DIR / "ASVspoof2019.LA.cm.eval.trl.txt",
}
AUDIO_DIRS = {
    "train": LA_ROOT / "ASVspoof2019_LA_train" / "flac",
    "dev": LA_ROOT / "ASVspoof2019_LA_dev" / "flac",
    "eval": LA_ROOT / "ASVspoof2019_LA_eval" / "flac",
}

for split in ["train", "dev", "eval"]:
    assert PROTOCOLS[split].exists(), f"Missing protocol: {PROTOCOLS[split]}"
    assert AUDIO_DIRS[split].exists(), f"Missing audio dir: {AUDIO_DIRS[split]}"

print(f"Repository root: {REPO_ROOT}")
print(f"ASVspoof 2019 LA root: {LA_ROOT}")

In [ ]:
def load_protocol(split: str) -> pd.DataFrame:
    df = pd.read_csv(
        PROTOCOLS[split],
        sep=r"\s+",
        header=None,
        names=["speaker_id", "utt_id", "unused", "attack_id", "label"],
    )
    df["split"] = split
    df["audio_path"] = df["utt_id"].map(lambda u: AUDIO_DIRS[split] / f"{u}.flac")
    return df

protocol_df = pd.concat([load_protocol(s) for s in ["train", "dev", "eval"]], ignore_index=True)

summary = (
    protocol_df.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "dev", "eval"])
)
summary["total"] = summary.sum(axis=1)
summary["spoof_ratio"] = (summary["spoof"] / summary["bonafide"]).round(2)
print("Official Partition Counts:")
display(summary)

# Verify a sample of audio files exist on disk
sample_paths = protocol_df.sample(n=min(100, len(protocol_df)), random_state=42)["audio_path"]
missing_files = [p for p in sample_paths if not p.exists()]
assert not missing_files, f"Missing audio files, e.g. {missing_files[:3]}"
print(f"Verified {len(sample_paths)} sampled FLAC files exist on disk.")

attack_counts = pd.crosstab(protocol_df["attack_id"], protocol_df["split"])[["train", "dev", "eval"]].reindex(
    ["-"] + [f"A{i:02d}" for i in range(1, 20)], fill_value=0
)
print("\nCounts by Attack ID ('-' = bona fide):")
display(attack_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# Left: Bona fide vs Spoof by split
summary[["bonafide", "spoof"]].plot(kind="bar", ax=axes[0], color=["#2B6CB0", "#C53030"])
axes[0].set_title("ASVspoof 2019 LA: Class Counts by Split")
axes[0].set_xlabel("Partition")
axes[0].set_ylabel("Number of Utterances")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)

# Right: Spoof attack ID counts across splits
spoof_only = attack_counts.drop(index="-", errors="ignore")
spoof_only.plot(kind="bar", stacked=True, ax=axes[1], color=["#4A5568", "#718096", "#DD6B20"])
axes[1].set_title("Spoof Attack IDs (A01-A06 Train/Dev, A07-A19 Eval)")
axes[1].set_xlabel("Attack ID")
axes[1].set_ylabel("Number of Utterances")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
out_fig1 = RESULTS_DIR / "eda_class_attack_distribution.png"
plt.savefig(out_fig1, dpi=150)
plt.show()
print(f"Saved: {out_fig1}")

In [ ]:
import librosa
import soundfile as sf
from tqdm.auto import tqdm

TARGET_SR = 16000
TOP_DB = 30
SAMPLE_PER_CLASS = 300
SEED = 42

train_df = protocol_df[protocol_df["split"] == "train"]
sample_df = pd.concat([
    train_df[train_df["label"] == "bonafide"].sample(n=SAMPLE_PER_CLASS, random_state=SEED),
    train_df[train_df["label"] == "spoof"].sample(n=SAMPLE_PER_CLASS, random_state=SEED),
], ignore_index=True)

records = []
for row in tqdm(sample_df.itertuples(), total=len(sample_df), desc="Measuring silence edges"):
    audio, sr = sf.read(row.audio_path, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    _, idx = librosa.effects.trim(audio, top_db=TOP_DB)
    total_sec = len(audio) / sr
    lead_sec = idx[0] / sr
    trail_sec = (len(audio) - idx[1]) / sr
    active_sec = (idx[1] - idx[0]) / sr

    records.append({
        "utt_id": row.utt_id,
        "label": row.label,
        "attack_id": row.attack_id,
        "total_sec": total_sec,
        "lead_silence_sec": lead_sec,
        "trail_silence_sec": trail_sec,
        "edge_silence_sec": lead_sec + trail_sec,
        "active_sec": active_sec,
    })

sil_df = pd.DataFrame(records)
print(f"Silence summary on {len(sil_df)} sampled Train clips (top_db={TOP_DB}):")
display(sil_df.groupby("label")[["total_sec", "lead_silence_sec", "trail_silence_sec", "edge_silence_sec", "active_sec"]].describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in [
    (axes[0], "lead_silence_sec", "Leading Silence (s)"),
    (axes[1], "trail_silence_sec", "Trailing Silence (s)"),
    (axes[2], "active_sec", "Active Speech After Trim (s)"),
]:
    for lbl, color in [("bonafide", "#2B6CB0"), ("spoof", "#C53030")]:
        vals = sil_df.loc[sil_df["label"] == lbl, col]
        ax.hist(vals, bins=25, alpha=0.55, label=lbl, color=color, density=True)
    ax.set_title(title)
    ax.set_xlabel("Seconds")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.25)

plt.suptitle(f"ASVspoof 2019 LA Train Sample (n={len(sil_df)}, top_db={TOP_DB})")
plt.tight_layout()
out_fig2 = RESULTS_DIR / "eda_silence_distributions.png"
plt.savefig(out_fig2, dpi=150)
plt.show()
print(f"Saved: {out_fig2}")

In [ ]:
ex_bonafide = sample_df[sample_df["label"] == "bonafide"].iloc[0]
ex_spoof = sample_df[sample_df["label"] == "spoof"].iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(13, 6.5))
for row_idx, ex in enumerate([ex_bonafide, ex_spoof]):
    y, sr = sf.read(ex["audio_path"], dtype="float32")
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    _, idx = librosa.effects.trim(y, top_db=TOP_DB)
    t = np.arange(len(y)) / sr

    # Waveform with trimmed silence edges shaded
    ax_w = axes[row_idx, 0]
    ax_w.plot(t, y, linewidth=0.7, color="#1A202C")
    ax_w.axvspan(0, idx[0] / sr, color="#CBD5E0", alpha=0.6, label="Trimmed edge")
    ax_w.axvspan(idx[1] / sr, len(y) / sr, color="#CBD5E0", alpha=0.6)
    ax_w.set_title(f"{ex['label'].upper()} ({ex['attack_id']}) Waveform: {ex['utt_id']}")
    ax_w.set_xlabel("Time (s)")
    ax_w.set_ylabel("Amplitude")
    ax_w.legend(loc="upper right", fontsize=8)

    # Log-Mel spectrogram
    ax_s = axes[row_idx, 1]
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=512, hop_length=160, n_mels=64)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    img = ax_s.imshow(
        log_mel,
        aspect="auto",
        origin="lower",
        extent=[0, len(y) / sr, 0, sr / 2],
        cmap="viridis",
    )
    ax_s.set_title(f"{ex['label'].upper()} ({ex['attack_id']}) Log-Mel Spectrogram")
    ax_s.set_xlabel("Time (s)")
    ax_s.set_ylabel("Frequency (Hz)")
    fig.colorbar(img, ax=ax_s, format="%+.0f dB")

plt.tight_layout()
out_fig3 = RESULTS_DIR / "eda_sample_waveform_spectrogram.png"
plt.savefig(out_fig3, dpi=150)
plt.show()
print(f"Saved: {out_fig3}")

### Check-In: Explain, Modify, Debug
- **Explain:** `setup.ipynb` verifies the environment, reads official ASVspoof 2019 LA protocols without mixing splits, and measures whether bona fide vs. spoof clips differ in leading/trailing silence (`top_db=30`).
- **Modify:** Adjust `TOP_DB`, `SAMPLE_PER_CLASS`, or `SEED` in the silence EDA cell to test threshold sensitivity on Dev clips before finalizing `src/` preprocessing.
- **Debug:** If `ASVSPOOF2019_LA_ROOT` fails, confirm the folder contains `ASVspoof2019_LA_cm_protocols/` and `ASVspoof2019_LA_train/flac/`.